In [2]:
import numpy as np
import matplotlib.pyplot as plt
from time import sleep
from pydaqmx_helper.dac import DAC

ModuleNotFoundError: No module named 'pydaqmx_helper'

In [2]:
# Defining path to folder raw data is saved in
path = "C:/Users/Student/Desktop/Robert-Kelly-Nuvistor-Triode/"

#### Setting up the picoammeter

In [165]:
# Import pyserial and set up a connection to the picoammeter.
# # [From physicslabs.ucd.ie RBD 910 Picoammeter documentation.]
import serial

ser = serial.Serial(
    "com5",
    baudrate = 57600,                    # Sets the bits per second to 57600
    bytesize = serial.EIGHTBITS,         # Sets the data bits to 8
    parity = serial.PARITY_NONE,         # Sets the parity to None
    stopbits = serial.STOPBITS_ONE,      # Sets the stop bits to 1
    xonxoff = False,                     # Sets the flow control to None
    timeout = 1,                         # Sets the timeout to 1 second
)

In [101]:
# Must talk to the picoammeter in arrays of bytes.
# data() turns a string "command" to be sent to the picoammeter into bytes 
# and also ends the string with the newline character "\n" so it does not have to be done manually
# # [From physicslabs.ucd.ie RBD 910 Picoammeter documentation.]

def data(command):
    return bytes(command+"\n", "utf-8")

def communicate(command): # This communicates the command to the picoammeter and returns its answer.
    ser.write(data(command))
    output = ser.readlines()
    print(output)

In [102]:
# Explicitly set filter.
communicate("&F064")

[b'&F, Filter=064\r\n']


In [96]:
communicate("&S")


[b'&S=,Range=002nA,-0.0025,nA\r\n']

In [ ]:
ser.write(data("&s01"))
ser.readline()

#### Define range of voltages to be sent to the circuit

The USB-6008 ranges from 0-5V. We want it to range from negative voltages to positive voltages so the NIDAQ analogue output adapter changes an input of 0V to -5V, 2.5V to 0V, and 5V to 5V.

In [31]:
# Write function to convert the actual voltage we want in the circuit to the value we tell the USB-6008 to transmit.
def convert_voltage(V):
    return V/2 + 2.5

The below convert function is written that will convert the current output given into usable 

In [166]:

def convert(command): 
    ser.write(data(command))    # When the command is "&S", the picoammeter gives back a list with a single byte inside.
    output = ser.readline()                  # Let output be that list with the single byte.
    output = output.decode("utf-8").rstrip() # decode the byte into a string and strip the newline character
    output = output.split(",")               # split the string into a list of 4 strings
    
    # Extracting the current from the output and accounting for the scale.
    
    if output[3] == "nA":
        # The third element of output is the current in nano-amps
        current = float(output[2]) * 10**-9
    
    if output[3] == "uA":
        # The third element of output is the current in micro-amps
        current = float(output[2]) *10**-6
        
    if output[3] == "uA":
        # The third element of output is the current in milli-amps
        current = float(output[2]) *10**-3
    
    I_A = np.append(I_A, current)

In [126]:

convert("&S")
current

NameError: name 'current' is not defined

In [173]:
# The numpy array V_A is the array of anode voltages applied to the circuit over the experimental run
# Range from -1 to 1V to test that the circuitry works.
# But USB-6008 only accepts from 0V to 5V therefore [-1V, 1V] -> [2V, 3V]
V_A = convert_voltage(np.arange(-1.5, 1, .025))

# Specify to send voltages using channel 0 on the USB-6008
myDAC = DAC(0)

# The time in seconds that the code will be delayed before moving to the next voltage in the range.
delay = .5

# empty array for recording the anode currents I_A.
I_A = np.array([])


alloutputs = []


# Loop over all voltages in the range
for V in V_A:
    
    # Send each voltage in the range to the circuit via the USB-6008
    myDAC.writeVoltage(V)
    
    #After each voltage in, wait before moving onto the next voltage in the range.
    sleep(delay)
    
    # Measure surrent from the picoammeter
    ser.write(data("&S"))    # When the command is "&S", the picoammeter gives back a list with a single byte inside.
    output = ser.readline()                  # Let output be that list with the single byte.
    
    
    
    alloutputs.append(output)
    
    
    
    output = output.decode("utf-8").rstrip() # decode the byte into a string and strip the newline character
    output = output.split(",")               # split the string into a list of 4 strings
    
    # Extracting the current from the output and accounting for the scale.
    
    if output[3] == "nA":
        # Then the third element of output is the current in nano-amps
        current = float(output[2]) * 10**-9
    
    if output[3] == "uA":
        # Then the third element of output is the current in micro-amps
        current = float(output[2]) *10**-6
        
    if output[3] == "mA":
        # Then the third element of output is the current in milli-amps
        current = float(output[2]) *10**-3
    

    I_A = np.append(I_A, current)

In [174]:
print(I_A)
print(alloutputs)

[-2.4973e-08 -2.4921e-06 -1.7761e-04 -1.7650e-04 -1.7588e-04 -1.7540e-04
 -1.7494e-04 -1.7444e-04 -1.7396e-04 -1.7347e-04 -1.7298e-04 -1.7248e-04
 -1.7201e-04 -1.7156e-04 -1.7112e-04 -1.7071e-04 -1.7030e-04 -1.6985e-04
 -1.6926e-04 -1.6820e-04 -1.6583e-04 -1.5910e-04 -1.4336e-04 -1.2067e-04
 -9.6300e-05 -7.3210e-05 -5.5170e-05 -4.0200e-05 -2.8530e-05 -1.9870e-05
 -1.2611e-05 -9.2760e-06 -6.4960e-06 -4.4230e-06 -3.0110e-06 -2.0250e-06
 -1.1243e-06 -9.2890e-07 -6.5360e-07 -4.4660e-07 -3.0020e-07 -2.0090e-07
 -7.0080e-08 -8.0790e-08 -5.8910e-08 -4.0370e-08 -2.6990e-08 -1.7980e-08
 -9.4900e-09 -7.8260e-09 -5.4770e-09 -3.6960e-09 -2.4820e-09 -1.6730e-09
 -9.2920e-10 -7.6360e-10 -5.3650e-10 -3.6740e-10 -2.5140e-10 -1.7010e-10
 -1.1690e-10 -7.7900e-11 -5.3000e-11 -3.6800e-11 -2.5100e-11 -1.8400e-11
 -1.5600e-11 -9.9000e-12 -9.9000e-12 -7.7000e-12 -6.4000e-12 -4.6000e-12
  3.0000e-13 -4.5000e-12 -6.0000e-13 -1.8000e-12 -1.6000e-12 -1.8000e-12
 -1.9000e-12 -1.1000e-12 -2.0000e-12 -1.1000e-12 -1

In [175]:
print(alloutputs)

[b'&S*,Range=020nA,-24.973,nA\r\n', b'&S*,Range=002uA,-2.4921,uA\r\n', b'&S*,Range=200uA,-177.61,uA\r\n', b'&S=,Range=200uA,-176.50,uA\r\n', b'&S=,Range=200uA,-175.88,uA\r\n', b'&S=,Range=200uA,-175.40,uA\r\n', b'&S=,Range=200uA,-174.94,uA\r\n', b'&S=,Range=200uA,-174.44,uA\r\n', b'&S=,Range=200uA,-173.96,uA\r\n', b'&S=,Range=200uA,-173.47,uA\r\n', b'&S=,Range=200uA,-172.98,uA\r\n', b'&S=,Range=200uA,-172.48,uA\r\n', b'&S=,Range=200uA,-172.01,uA\r\n', b'&S=,Range=200uA,-171.56,uA\r\n', b'&S=,Range=200uA,-171.12,uA\r\n', b'&S=,Range=200uA,-170.71,uA\r\n', b'&S=,Range=200uA,-170.30,uA\r\n', b'&S=,Range=200uA,-169.85,uA\r\n', b'&S=,Range=200uA,-169.26,uA\r\n', b'&S=,Range=200uA,-168.20,uA\r\n', b'&S=,Range=200uA,-165.83,uA\r\n', b'&S=,Range=200uA,-159.10,uA\r\n', b'&S=,Range=200uA,-143.36,uA\r\n', b'&S=,Range=200uA,-120.67,uA\r\n', b'&S=,Range=200uA,-096.30,uA\r\n', b'&S=,Range=200uA,-073.21,uA\r\n', b'&S=,Range=200uA,-055.17,uA\r\n', b'&S=,Range=200uA,-040.20,uA\r\n', b'&S=,Range=200uA,-

In [3]:
plt.plot(V_A, I_A, "o")
plt.xlabel("Applied anode voltage (V)", fontsize = 12)
plt.ylabel("Anode current (A)", fontsize = 12)
plt.title("I-V curve of anode current versus applied anode voltage in the nuvistor triode", fontsize = 12);

NameError: name 'V_A' is not defined

In [1]:
# Load in raw data if it has been saved already.
I_A = np.load(path+"anode currents.npy")
V_A = np.load(path+"anode voltages.npy")
print(I_A, V_A)

I_A = np.load("C:/Users/robtk/OneDrive/Desktop/3rd year Labs/Nuvistor Triode/anode currents.npy")
V_A = np.load("C:/Users/robtk/OneDrive/Desktop/3rd year Labs/Nuvistor Triode/anode currents.npy")

NameError: name 'np' is not defined

In [177]:
# Save raw data to a file so it can be accessed later.
np.save(path+"anode currents", I_A)
np.save(path+"anode voltages", V_A)